In [1]:
from ultralytics import YOLO

model_weights_path = "../weights/yolo-seg/yolo11s_giftbox.pt"

model = YOLO(model_weights_path)


## Predict the image folder 

In [2]:
from pathlib import Path

image_path = Path("../assets/giftbox_task/images")

predict_results = model(image_path)

demo_pred = predict_results[0]

print(demo_pred.masks)




image 1/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_1.png: 480x640 1 red box, 1 green small box, 35.3ms
image 2/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_10.png: 480x640 1 red box, 1 small object, 1 green small box, 1 toy car, 5.2ms
image 3/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_100.png: 480x640 1 red box, 1 small object, 1 green small box, 1 toy car, 4.8ms
image 4/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_101.png: 480x640 1 red box, 1 small object, 1 green small box, 1 toy car, 4.8ms
image 5/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_102.png: 480x640 1 red box, 1 green small box, 4.4ms
image 6/228 /home/yy/yy/github/automatic-annotation/src/../assets/giftbox_task/images/giftbox_103.png: 480x640 1 red box, 1 small object, 1 green small box, 4.8ms
image 7/228 /home/yy/y

In [3]:
print(demo_pred.masks)

ultralytics.engine.results.Masks object with attributes:

data: tensor([[[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]], device='cuda:0', dtype=torch.uint8)
orig_shape: (480, 640)
shape: torch.Size([2, 480, 640])
xy: [array([[        282,         122],
       [        282,         126],
       [        279,         129],
       [        279,         130],
       [        278,         131],
       [        278,         132],
       [        271,         139],
       [        270,         139],
       [        267,         142],
       [        266,         142],
       [        265,         143],
  

In [4]:
import json
from pathlib import Path
from skimage import measure
import numpy as np



def make_coco_annotation(predict_results, model, save_path):
    """
    Convert YOLO segmentation predict results to COCO format annotation.json
    
    Args:
        predict_results: list of ultralytics.engine.results.Results
        model: YOLO model (for model.names)
        save_path: Path to save the annotation.json
    """
    class_names = model.names  # {0: 'class0', 1: 'class1', ...}
    
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": int(cat_id), "name": name, "supercategory": "object"}
            for cat_id, name in class_names.items()
        ],
    }
    
    ann_id = 1
    
    for img_id, result in enumerate(predict_results, start=1):
        height, width = result.orig_shape
        image_path = Path(result.path)
        
        coco_output["images"].append({
            "id": img_id,
            "file_name": image_path.name,
            "width": width,
            "height": height,
        })
        
        if result.masks is None:
            print(f"  ⚠ No masks found for {image_path.name}, skipping...")
            continue
        
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        class_ids = result.boxes.cls.cpu().numpy().astype(int)
        confidences = result.boxes.conf.cpu().numpy()
        
        for i, mask_polygon in enumerate(result.masks.xy):
            x1, y1, x2, y2 = boxes_xyxy[i]
            w = x2 - x1
            h = y2 - y1
            
            segmentation = mask_polygon.flatten().tolist()
            
            poly = mask_polygon
            area = 0.5 * abs(np.dot(poly[:, 0], np.roll(poly[:, 1], 1)) -
                            np.dot(poly[:, 1], np.roll(poly[:, 0], 1)))
            
            coco_output["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": int(class_ids[i]),
                "segmentation": [segmentation],
                "bbox": [float(x1), float(y1), float(w), float(h)],
                "area": float(area),
                "iscrowd": 0
            })
            ann_id += 1
    
    # Save
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:

        for cat_id in sorted(class_names.keys()):
            f_labels.write(f"{class_names[cat_id]}\n")
    
    num_imgs = len(coco_output["images"])
    num_anns = len(coco_output["annotations"])
    num_cats = len(coco_output["categories"])
    print(f"✅ COCO annotation saved to {save_path}")
    print(f"   images={num_imgs}, annotations={num_anns}, categories={num_cats}")




In [5]:

annotation_file = Path("../assets/giftbox_task/annotations/annotations.json")

if not annotation_file.exists():
    annotation_file.parent.mkdir(parents=True, exist_ok=True)
    annotation_file.touch()


make_coco_annotation(predict_results, model, save_path=annotation_file)

✅ COCO annotation saved to ../assets/giftbox_task/annotations/annotations.json
   images=228, annotations=696, categories=4
